# A股主板周频多因子 v0.1

2万元 · 免费数据 · 周频 · 手动下单 · 15%回撤警戒线

**默认模拟数据，仅验证代码。真实行情下载与收益尚未在交付环境验证；研究回测不等于整数手实盘收益。**
先将本Notebook上传到Colab，第一格按提示上传 `ashare_colab.zip`。完整假设和限制见项目README。

In [ ]:
from pathlib import Path
import os, sys, zipfile
root = Path('/content/ashare_weekly') if Path('/content').exists() else Path.cwd()
if not (root / 'ashare').exists():
    from google.colab import files
    uploaded = files.upload()  # 选择 ashare_colab.zip
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('请只上传一个项目ZIP')
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.namelist():
            target = (destination / member).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Unsafe archive path')
        z.extractall(destination)
os.chdir(root)
sys.path.insert(0, str(root))
print('工作目录:', root)


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## 数据

`MODE='demo'` 是模拟数据。`MODE='free'` 下载真实数据，无需密钥。
先用30只验证接口；这不是代表性股票池。完整研究改 `MAX_SYMBOLS=None`，下载会按股票缓存。
长时间下载前，可挂载自己的Google Drive，并把 `CACHE_DIR` 改成个人Drive下的目录。

In [ ]:
from ashare.data import load_free
from ashare.demo import synthetic_data
MODE = 'demo'  # 验证接口时改为 'free'
MAX_SYMBOLS = 30  # 全市场研究改为 None；30只按代码选取，有样本偏差
CACHE_DIR = 'data/baostock'
if MODE == 'demo':
    prices, fundamentals, benchmark, metadata = synthetic_data()
elif MODE == 'free':
    prices, fundamentals, benchmark, metadata = load_free(
        start='2017-01-01', end='2025-12-31', cache_dir=CACHE_DIR, max_symbols=MAX_SYMBOLS)
else:
    raise ValueError('Unknown MODE')
print(metadata)
print('行情行数:', len(prices), '财报行数:', len(fundamentals))
# 数据本身的首末日期和股票覆盖，不能只看API是否成功。
print(prices.groupby('code').date.agg(['min','max']).head(10))


## 固定基线与分期

开发期2019–2021、验证期2022–2023、留出期2024–2025。
先在开发期检查假设，再评估验证期；留出期开关默认关闭。不要看完留出结果后继续针对它调参数。
所有结果均为连续份额、分红再投资假设下的研究结果。

In [ ]:
from dataclasses import replace
from datetime import datetime
import pandas as pd
from ashare.strategy import Config
from ashare.research import backtest, save_run
cfg = Config(drawdown_alert=.15)
BASELINES = {
    'momentum': replace(cfg, value_weight=0, quality_weight=0, momentum_weight=1),
    'value_quality': replace(cfg, value_weight=.5, quality_weight=.5, momentum_weight=0),
    'composite': cfg,
    'composite_no_trend': replace(cfg, trend_filter=False),
}
# 首次演示使用验证期；真实研究先切换至开发期。
PERIOD = ('2022-01-01', '2023-12-31')
run_root = Path('outputs') / (MODE + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
results = {}
for name, candidate in BASELINES.items():
    result = backtest(prices, fundamentals, benchmark, candidate, *PERIOD)
    save_run(result, run_root / name, metadata)
    results[name] = result
comparison = pd.DataFrame({name: {k: r[3][k] for k in (
    'total_return', 'annualized_return', 'max_drawdown', 'sharpe_rf0',
    'estimated_cost', 'benchmark_return', 'drawdown_alert_breached')}
    for name, r in results.items()}).T
print('模式:', MODE, '；收益不代表真实整数手账户；输出:', run_root)
display(comparison)
comparison.to_csv(run_root / 'comparison.csv')


In [ ]:
os.environ.setdefault('MPLCONFIGDIR', '/tmp/ashare_matplotlib')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for name, (curve, _, _, report) in results.items():
    axes[0].plot(curve.index, curve.nav / cfg.capital, label=name)
    axes[1].plot(curve.index, curve.nav / curve.nav.cummax().clip(lower=cfg.capital) - 1, label=name)
b = benchmark.loc[results['composite'][0].index]
axes[0].plot(b.index, b / b.iloc[0], '--', label='HS300 price index')
axes[0].set_title(MODE.upper() + ' — RESEARCH ONLY, fractional positions')
axes[0].legend(); axes[1].axhline(-cfg.drawdown_alert, color='red', linestyle='--')
axes[1].set_ylabel('Drawdown'); plt.tight_layout()
fig.savefig(run_root / 'nav_drawdown.png', dpi=140)
plt.show()
print('检查每次实验目录中的 annual_returns.csv、trades.csv、skipped.csv、report.json。')


## 因子诊断

每周截面Rank IC和五分组收益：使用下一周首个开盘到再下一周首个开盘的复权收益。
这是事后诊断，不是信号输入；不含费用、整数手或成交约束。样本不足10只则报错，不输出空的“成功”结果。

In [ ]:
from ashare.diagnostics import factor_diagnostics
detail, factor_summary = factor_diagnostics(prices, fundamentals, benchmark, cfg, *PERIOD)
display(factor_summary)
detail.to_csv(run_root / 'factor_diagnostics.csv', index=False)
factor_summary.to_csv(run_root / 'factor_summary.csv')


## 留出期（默认关闭）

只有在参数及筛选规则冻结后运行。若曾查看或使用过该期间，不能再称其为未见样本。
不要按这次表现自动挑选赢家。真实数据与完整股票池是必要条件，但不代表数据偏差已消除。

In [ ]:
RUN_FINAL_HOLDOUT = False
if RUN_FINAL_HOLDOUT:
    if MODE != 'free' or metadata.get('sample_only', True):
        raise ValueError('留出评估需要真实数据和完整股票池')
    final = backtest(prices, fundamentals, benchmark, cfg, '2024-01-01', '2025-12-31')
    save_run(final, run_root / 'final_holdout', metadata)
    print(final[3])


## 调仓清单：先演示，真实持仓必须自行提供

下面是模拟持仓清单，不是投资建议。实盘/模拟盘模式需准备：
- `holdings.csv`：code,shares,sellable（券商实际持仓与可卖数量）
- `quotes.csv`：code,price,upper,lower,tradable（当日最新报价与准确涨跌停价）
- 显式填写可用现金、已完成的信号日期、计划交易日期。

当前下载样本截止2025年，**不能用于生成今天的实盘清单**。真实模式需先用最新已完成交易日重新下载；不得拿历史回测持仓当实际持仓。只使用已完成信号，交易日期必须更晚。预计卖单收入不计入买入预算。

In [ ]:
from ashare.strategy import prepare_prices, targets
from ashare.orders import plan_orders
ORDER_MODE = 'demo'  # 实际持仓演练改 'manual'；不会自动下单
if ORDER_MODE == 'demo':
    dp, df, db, _ = synthetic_data()
    SIGNAL_DATE = pd.Timestamp('2023-12-29')
    TRADE_DATE = pd.Timestamp('2024-01-02')
    chosen = targets(prepare_prices(dp), df, db, SIGNAL_DATE, replace(cfg, trend_filter=False))
    raw = dp[dp.date.eq(SIGNAL_DATE)].set_index('code')
    quotes = pd.DataFrame({'price':raw.close, 'upper':(raw.close*1.10).round(2),
                           'lower':(raw.close*.90).round(2), 'tradable':True})
    holdings = pd.DataFrame(columns=['code','shares','sellable'])
    cash = 20000
elif ORDER_MODE == 'manual':
    if MODE != 'free' or metadata.get('sample_only', True):
        raise ValueError('需要最新真实数据和完整股票池')
    # 修改以下日期及现金；还需先按新截止日期更新prices/fundamentals/benchmark。
    SIGNAL_DATE = None
    TRADE_DATE = None
    cash = None
    if any(x is None for x in (SIGNAL_DATE, TRADE_DATE, cash)):
        raise ValueError('请显式填写SIGNAL_DATE、TRADE_DATE和券商可用现金')
    SIGNAL_DATE, TRADE_DATE = pd.Timestamp(SIGNAL_DATE), pd.Timestamp(TRADE_DATE)
    today = pd.Timestamp.now(tz='Asia/Shanghai').tz_localize(None).normalize()
    if SIGNAL_DATE >= TRADE_DATE or SIGNAL_DATE >= today or TRADE_DATE < today or (TRADE_DATE-SIGNAL_DATE).days>10:
        raise ValueError('信号必须是已完成交易日，且不能用过期信号生成历史清单')
    if SIGNAL_DATE != benchmark.index[-1]:
        raise ValueError('请先更新至最新已完成的信号交易日数据')
    holdings = pd.read_csv('holdings.csv', dtype={'code':str})
    quotes = pd.read_csv('quotes.csv', dtype={'code':str}).set_index('code')
    chosen = targets(prepare_prices(prices), fundamentals, benchmark, SIGNAL_DATE, cfg)
else:
    raise ValueError('Unknown ORDER_MODE')
orders, cash_check = plan_orders(chosen, holdings, quotes, cash, TRADE_DATE, cfg)
print('示例' if ORDER_MODE == 'demo' else '人工审核清单；尚未下单', cash_check)
display(orders)
orders.to_csv(run_root / (ORDER_MODE + '_orders.csv'), index=False)


## 导出

实验目录包含参数、数据标记、指标、净值、交易与诊断。保留所有版本，不只保留收益最好的一次。
免费数据的修订和退市覆盖尚未审计；此版本没有行业约束及逐股分红/整数手真实成交回测，进入实盘前还需完成这些工作及前向模拟。

In [ ]:
import shutil
archive = shutil.make_archive(str(run_root), 'zip', run_root)
print('已导出:', archive)
if Path('/content').exists():
    from google.colab import files
    files.download(archive)
